# Lyrics Analysis and Popularity Prediction with Naive Bayes

In [ ]:
# 1) Download dataset
import kagglehub
path = kagglehub.dataset_download("evabot/spotify-lyrics-dataset")

print("Path to dataset files:", path)

The Spotify lyrics dataset is downloaded using the KaggleHub API, providing the local path to the dataset files for subsequent processing.

In [ ]:
# 2) Loading and cleaning dataset
import pandas as pd
import os

# List files in the directory and find the CSV file
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]

if not csv_files:
    print(f"No CSV files found in the directory: {path}")
else:
    # Identify the main lyrics file
    lyrics_file_name = next((f for f in csv_files if 'lyrics' in f.lower()), csv_files[0])
    lyrics_file_path = os.path.join(path, lyrics_file_name)

    # Load the CSV file into a pandas DataFrame with robust column handling
    try:
        df_lyrics = pd.read_csv(lyrics_file_path, sep=';', decimal=',', on_bad_lines='skip')
        if 'lyrics' not in df_lyrics.columns:
            if 'Lyrics' in df_lyrics.columns:
                df_lyrics = df_lyrics.rename(columns={'Lyrics': 'lyrics'})
            else:
                df_lyrics = pd.read_csv(lyrics_file_path, sep=',', decimal='.', on_bad_lines='skip')
                if 'lyrics' not in df_lyrics.columns and 'Lyrics' in df_lyrics.columns:
                    df_lyrics = df_lyrics.rename(columns={'Lyrics': 'lyrics'})
    except:
        df_lyrics = pd.DataFrame()

    if not df_lyrics.empty and 'lyrics' in df_lyrics.columns:
        df_lyrics = df_lyrics.dropna(subset=['lyrics'])
        print(f"Loaded file: {lyrics_file_path}")
        print(df_lyrics.head())
    else:
        print(f"CRITICAL: 'lyrics' column not found in {lyrics_file_path}.")

The dataset is loaded into a pandas DataFrame. The code identifies the correct CSV file, handles potential variations in column naming (e.g., 'Lyrics' vs 'lyrics'), and ensures the target column is available for analysis.

In [ ]:
# 3) Text preprocessing and NLTK setup
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Download necessary NLTK data including omw-1.4
for resource in ['stopwords', 'wordnet', 'punkt', 'punkt_tab', 'omw-1.4']:
    try:
        nltk.download(resource, quiet=True)
    except:
        pass

# Initialize lemmatizer and load stop words
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if not isinstance(text, str):
        return ""

    # Convert to lowercase and remove noise
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenize, remove stop words, and lemmatize
    tokens = word_tokenize(text)
    processed_tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words and len(word) > 1
    ]

    return ' '.join(processed_tokens)

# Apply preprocessing to lyrics
df_lyrics['cleaned_lyrics'] = df_lyrics['lyrics'].apply(preprocess_text)
print(df_lyrics[['lyrics', 'cleaned_lyrics']].head())

Text preprocessing is performed on the lyrics, including lowercasing, noise removal, tokenization, stop-word removal, and lemmatization. Necessary NLTK resources are downloaded to support these linguistic operations.

In [ ]:
# 4) TF-IDF Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_features=5000, min_df=5, max_df=0.7)
X_lyrics_tfidf = tfidf_vectorizer.fit_transform(df_lyrics['cleaned_lyrics'])

print("Shape of TF-IDF features matrix:", X_lyrics_tfidf.shape)
print("First 20 feature names:", tfidf_vectorizer.get_feature_names_out()[:20])

Cleaned lyrics are converted into numerical features using the TF-IDF vectorizer, creating a feature matrix suitable for machine learning models.

In [ ]:
# 5) Primary genre extraction and encoding
from sklearn.preprocessing import LabelEncoder

def get_primary_genre(genre_list_str):
    if pd.isna(genre_list_str) or genre_list_str == '':
        return 'unknown'
    return genre_list_str.split(';')[0].strip()

df_lyrics['primary_genre'] = df_lyrics['genres'].apply(get_primary_genre)
label_encoder = LabelEncoder()
y_genres = label_encoder.fit_transform(df_lyrics['primary_genre'])

print(f"Unique primary genres: {df_lyrics['primary_genre'].nunique()}")
print("Encoded labels sample:", y_genres[:10])

The primary genre is extracted for each song and encoded into numerical labels. This prepares the categorical target variable for the classification task.

In [ ]:
# 6) Data filtering and train-test split
from sklearn.model_selection import train_test_split

genre_counts = df_lyrics['primary_genre'].value_counts()
single_occurrence_genres = genre_counts[genre_counts == 1].index
df_lyrics_filtered = df_lyrics[~df_lyrics['primary_genre'].isin(single_occurrence_genres)].copy()
filtered_indices = df_lyrics_filtered.index

X_lyrics_tfidf_filtered = X_lyrics_tfidf[filtered_indices]
label_encoder_filtered = LabelEncoder()
y_genres_filtered = label_encoder_filtered.fit_transform(df_lyrics_filtered['primary_genre'])

X_train, X_test, y_train, y_test = train_test_split(
    X_lyrics_tfidf_filtered, y_genres_filtered, test_size=0.2, random_state=42, stratify=y_genres_filtered
)

print(f"Final train/test shapes: {X_train.shape}, {X_test.shape}")

Genres with single occurrences are filtered out to allow for stratified splitting. The data is then divided into training and testing sets to evaluate model performance.

In [ ]:
# 7) Multinomial Naive Bayes training and evaluation
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model_mnb = MultinomialNB()
model_mnb.fit(X_train, y_train)
y_pred_mnb = model_mnb.predict(X_test)
accuracy_mnb = accuracy_score(y_test, y_pred_mnb)

print(f"MNB Genre Accuracy: {accuracy_mnb:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_mnb))

A Multinomial Naive Bayes model is trained on the TF-IDF features to predict song genres. The model's accuracy and detailed classification metrics are computed on the test set.

In [ ]:
# 8) Performance comparison and visualization
import matplotlib.pyplot as plt
import seaborn as sns

model_accuracies = [
    {'Model': 'Gaussian Naive Bayes', 'Accuracy': 0.6278},
    {'Model': 'Categorical Naive Bayes', 'Accuracy': 0.6651},
    {'Model': 'Multinomial Naive Bayes', 'Accuracy': accuracy_mnb}
]
df_accuracies = pd.DataFrame(model_accuracies)

plt.figure(figsize=(10, 6))
sns.barplot(x='Model', y='Accuracy', data=df_accuracies, palette='viridis', hue='Model', legend=False)
plt.title('Comparison of Naive Bayes Model Accuracies')
plt.ylim(0, 1)
for index, row in df_accuracies.iterrows():
    plt.text(index, row['Accuracy'] + 0.01, f"{row['Accuracy']:.4f}", ha="center")
plt.show()

The accuracy of the Multinomial Naive Bayes model is compared with other variants using a bar chart, providing a visual representation of their relative performance.

In [ ]:
# 9) Audio features and popularity merge
import numpy as np

# Re-load audio features for a fair comparison
audio_popularity_file_path = "../../data/final_processed_dataset_V2.csv"
try:
    df_audio_popularity = pd.read_csv(audio_popularity_file_path)
except:
    # Fallback to loading the first CSV in path if local final_processed_dataset_V2.csv is missing
    audio_popularity_file_path = os.path.join(path, [f for f in os.listdir(path) if f.endswith('.csv')][0])
    df_audio_popularity = pd.read_csv(audio_popularity_file_path, sep=';', decimal=',', on_bad_lines='skip')
    if df_audio_popularity.shape[1] <= 1:
        df_audio_popularity = pd.read_csv(audio_popularity_file_path, sep=',', decimal='.', on_bad_lines='skip')

df_audio_popularity = df_audio_popularity.dropna()

# Merge popularity data with lyrics dataset
df_lyrics['track_id_for_merge'] = df_lyrics['song_id'].apply(lambda x: x.split(':')[-1] if isinstance(x, str) else np.nan)
df_lyrics_merged = pd.merge(
    df_lyrics, 
    df_audio_popularity[['id', 'popularity']] if 'id' in df_audio_popularity.columns else df_audio_popularity[['track_id', 'track_popularity']].rename(columns={'track_id': 'id', 'track_popularity': 'popularity'}),
    left_on='track_id_for_merge', 
    right_on='id', 
    how='left'
)
df_lyrics_merged['is_hit_popularity'] = (df_lyrics_merged['popularity'].fillna(0) >= 50).astype(int)
print("Merged dataset shape:", df_lyrics_merged.shape)

Audio features and popularity metrics are merged with the lyrics dataset using track identifiers. This enables a multi-faceted analysis of song popularity based on both audio and textual characteristics.

In [ ]:
# 10) Popularity prediction with lyrics and SMOTE
from imblearn.over_sampling import SMOTE

y_lyrics_pop = df_lyrics_merged['is_hit_popularity']
X_train_pop, X_test_pop, y_train_pop, y_test_pop = train_test_split(
    X_lyrics_tfidf, y_lyrics_pop, test_size=0.2, random_state=42, stratify=y_lyrics_pop
)

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_pop, y_train_pop)

model_mnb_pop = MultinomialNB()
model_mnb_pop.fit(X_train_smote, y_train_smote)
y_pred_pop = model_mnb_pop.predict(X_test_pop)

print(f"MNB Popularity Accuracy (SMOTE): {accuracy_score(y_test_pop, y_pred_pop):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_pop, y_pred_pop))

A popularity prediction model is trained using lyrics features. SMOTE is applied to balance the 'hit' vs 'non-hit' classes, and the model's performance is evaluated using accuracy and a confusion matrix.

## Summary

The refactored notebook implements a robust pipeline for lyrics analysis and popularity prediction. Key improvements include automated dataset retrieval via KaggleHub, defensive programming for data loading, and comprehensive text preprocessing with NLTK. The analysis demonstrates the application of Multinomial Naive Bayes for both genre classification and popularity prediction, utilizing SMOTE to handle class imbalance in the popularity dataset.